In [1]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

c:\Users\ansul\OneDrive\Desktop\data science project\financial_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## Load chunks
with open(r"..\data\chunked\chunks_type2_Semantic.json") as f:
    chunks = json.load(f)

In [3]:
## creating document object
documents = []

for c in chunks:
    doc = Document(
        page_content=c["text"],
        metadata={"company_name": c["company_name"], "section": c["section"], "chunk_id": c["chunk_id"]},
    )
    documents.append(doc)

In [4]:
## embedding model intialize
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en"
)


C:\Users\ansul\AppData\Local\Temp\ipykernel_11940\418201913.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2736.25it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Creating index file for first time

In [5]:
## building vector store
vectorstore = FAISS.from_documents(
    documents,
    embedding_model
)

vectorstore.save_local(r"..\data\embeddings_semantic_BAAI")

## Use the local vector index

In [10]:
## loading the vector store
vectorstore = FAISS.load_local(
    r"..\data\embeddings",embeddings=embedding_model,allow_dangerous_deserialization=True
)

## Testing Retrival 

In [7]:
vectorstore.similarity_search("how does companies stock price depend on global and regional economic conditions ?", k=3)

[Document(id='8692703a-4f2b-46e8-b0e4-3c11bf43afba', metadata={'company_name': 'ORACLE', 'section': 'risk_factors', 'chunk_id': 20}, page_content='30 Table of Contents Index to Financial Statements General Risks Economic, political and market conditions can adversely affect our business, results of operations and financial condition, including our revenue growth and profitability, which in turn could adversely affect our stock price. Our business is influenced by a range of factors that are beyond our control and that we have no comparative advantage in forecasting. These include: general economic and business conditions; overall demand for enterprise cloud, license and hardware products and services; governmental budgetary constraints or shifts in government spending priorities; and general legal, regulatory and political developments. Macroeconomic developments such as the global or regional economic effects resulting from increasing inflation rates, limited liquidity, adverse develo

In [8]:
vectorstore.similarity_search("What is the risks associated with netflix?", k=3)

[Document(id='e159daf2-f478-405c-b9d9-4fc003d700ec', metadata={'company_name': 'NFLX', 'section': 'risk_factors', 'chunk_id': 4}, page_content="From time to time, we acquire or invest in businesses, content, and technologies that support our business. The risks associated with such acquisitions or investments (some of which may be unforeseen) include the difficulty of integrating solutions, operations, and personnel; inheriting liabilities and exposure to litigation; failure to realize anticipated benefits and expected synergies; and diversion of managements time and attention, among other acquisition-related risks. We may not be successful in overcoming such risks, and such acquisitions and investments may negatively impact our business. In addition, if we do not complete an announced acquisition transaction or integrate an acquired business successfully and in a timely manner, we may not realize the benefits of the acquisition to the extent anticipated. Acquisitions and investments m